In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================
# 1. Load data
# =========================
solar_df = pd.read_csv("/content/Solar_Energy_Generation.csv.zip")
weather_df = pd.read_csv("/content/Weather_Data_reordered_all.csv.zip")
site_df = pd.read_csv("/content/Solar_Site_Details.csv")


# =========================
# 2. Convert timestamp
# =========================
solar_df['Timestamp'] = pd.to_datetime(solar_df['Timestamp'])
weather_df['Timestamp'] = pd.to_datetime(weather_df['Timestamp'])

# =========================
# 3. Merge solar + weather
# =========================
df = pd.merge(
    solar_df,
    weather_df,
    on=['CampusKey', 'Timestamp'],
    how='inner'
)

# =========================
# 4. Merge site details (numeric only)
# =========================
site_features = ['CampusKey', 'SiteKey', 'kWp', 'Number of panels', 'lat', 'Lon']
df = pd.merge(
    df,
    site_df[site_features],
    on=['CampusKey', 'SiteKey'],
    how='left'
)

# =========================
# 5. Basic cleaning
# =========================
df = df.drop_duplicates()
df = df.sort_values(['SiteKey', 'Timestamp'])

# =========================
# 6. Time-based feature engineering
# =========================
df['hour'] = df['Timestamp'].dt.hour
df['day'] = df['Timestamp'].dt.day
df['month'] = df['Timestamp'].dt.month
df['dayofweek'] = df['Timestamp'].dt.dayofweek

# Cyclical encoding
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# =========================
# 7. Weather-derived features
# =========================
df['temp_spread'] = df['AirTemperature'] - df['DewPointTemperature']
df['wind_humidity_interaction'] = df['WindSpeed'] * df['RelativeHumidity']

# Wind direction cyclical encoding (better than raw angle)
df['wind_dir_sin'] = np.sin(2 * np.pi * df['WindDirection'] / 360)
df['wind_dir_cos'] = np.cos(2 * np.pi * df['WindDirection'] / 360)

# =========================
# 8. Lag features (very useful)
# =========================
df['SolarGeneration_lag1'] = df.groupby('SiteKey')['SolarGeneration'].shift(1)
df['SolarGeneration_lag2'] = df.groupby('SiteKey')['SolarGeneration'].shift(2)
df['AirTemperature_lag1'] = df.groupby('SiteKey')['AirTemperature'].shift(1)

# =========================
# 9. Rolling features (optional but strong)
# =========================
df['SolarGeneration_roll3'] = df.groupby('SiteKey')['SolarGeneration'].transform(
    lambda x: x.rolling(3).mean()
)

# =========================
# 10. Handle missing values
# =========================
num_cols = [
    'ApparentTemperature', 'AirTemperature', 'DewPointTemperature',
    'RelativeHumidity', 'WindSpeed', 'WindDirection',
    'kWp', 'Number of panels', 'lat', 'Lon'
]

for col in num_cols:
    df[col] = df.groupby('CampusKey')[col].transform(lambda x: x.interpolate().ffill().bfill())

# Drop rows created as NaN due to lag/rolling
df = df.dropna()

# =========================
# 11. Optional: remove very low generation rows
# (You can comment this out initially)
# =========================
# df = df[df['SolarGeneration'] > 0]

# =========================
# 12. Define features and target
# =========================
target = 'SolarGeneration'

features = [
    'ApparentTemperature',
    'AirTemperature',
    'DewPointTemperature',
    'RelativeHumidity',
    'WindSpeed',
    'kWp',
    'Number of panels',
    'lat',
    'Lon',
    'hour',
    'day',
    'month',
    'dayofweek',
    'hour_sin',
    'hour_cos',
    'month_sin',
    'month_cos',
    'temp_spread',
    'wind_humidity_interaction',
    'wind_dir_sin',
    'wind_dir_cos',
    'SolarGeneration_lag1',
    'SolarGeneration_lag2',
    'AirTemperature_lag1',
    'SolarGeneration_roll3'
]

X = df[features]
y = df[target]

# =========================
# 13. Time-based split
# =========================
df = df.sort_values('Timestamp')
X = df[features]
y = df[target]

split_idx = int(len(df) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

# =========================
# 14. Train model
# =========================
model = RandomForestRegressor(
    n_estimators=150,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# =========================
# 15. Predict + evaluate
# =========================
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

# =========================
# 16. Feature importance
# =========================
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\nTop Feature Importances:")
print(importance_df.head(15))

: 